In [150]:
import importlib
import sys
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / 'src/allocations.py').exists()), Path.cwd())
sys.path.insert(0, str(ROOT))
import src.allocations
import src.metrics2

importlib.reload(src.allocations)
importlib.reload(src.metrics2)
from tqdm.auto import tqdm

from src.metrics2 import cohort_metrics

In [151]:
NEED_COL          = "gagne_sum_t"
EXP_NEED_COL      = "exp_need_{model}"
GROUP_COL         = "race"
METHOD            = "bootstrap"
MODEL_CLASS       = "nn_multiclass"
RASHOMON_EPS      = 0.01
ALLOCATION_EPS    = 0.1
N_MODELS          = 100
SAMPLE_SIZE       = 100
ITERS_PER_SEED    = 5
fp_input          = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"

PRED_METHODS      = ["average"]
NEED_THRESHOLDS   = [1]
SELECTION_RATES   = [0.1]
REPLACE_ALPHAS    = [0.0, 0.25, 0.5, 0.75, 1.0]
ALLOCATION_DIR    = ROOT / "data/results/allocations"
# Beta values are discovered from saved *_beta-<value>.csv pairs below.


In [152]:
fp_raw = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
df_raw = pd.read_csv(fp_raw)

In [153]:
cohort_metric_columns = [
    "pairwise_consistency",
    "systemic_exclusion_all",
    "systemic_exclusion_qualified",
]
prediction_cache = {}


def predictions_for(pred_method, need_threshold):
    key = (pred_method, need_threshold)
    if key not in prediction_cache:
        qualification_col = f"qualified_gagne_{need_threshold}"
        path = ROOT / f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{pred_method}_{qualification_col}.csv"
        predictions = pd.read_csv(path)
        prediction_cache[key] = predictions.merge(
            df_raw[["person_id", GROUP_COL]], on="person_id", how="left", validate="many_to_one"
        )
    return prediction_cache[key]


def allocation_tag(pred_method, need_threshold, selection_rate):
    return (
        f"{METHOD}_{MODEL_CLASS}_{pred_method}_y-qualified_gagne_{need_threshold}"
        f"_rate-{selection_rate}_size-{SAMPLE_SIZE}"
    )


def read_allocations(path):
    return pd.read_csv(path, converters={"selected": literal_eval, "models": literal_eval})


def beta_runs(tag, directory=ALLOCATION_DIR):
    """Discover saved beta runs; both allocations and cohort files must exist."""
    prefix = f"lda_exp_{tag}_beta-"
    runs = {}
    for allocation_path in directory.glob(f"{prefix}*.csv"):
        suffix = allocation_path.stem[len(prefix):]
        beta = float(suffix)
        if not np.isfinite(beta) or not 0 <= beta <= 1:
            raise ValueError(f"Invalid beta in {allocation_path.name}")
        people_path = directory / f"people_{tag}_beta-{suffix}.csv"
        if not people_path.exists():
            print(f"Skipping beta={beta:g}: cohort file is not available yet.")
            continue
        if beta in runs:
            raise ValueError(f"Multiple saved runs for beta={beta:g}: {tag}")
        runs[beta] = (people_path, allocation_path)
    return [(beta, *runs[beta]) for beta in sorted(runs)]


def evaluate_allocations(df, df_people, algorithm_allocations, *, pred_method,
                         need_threshold, selection_rate, alpha, beta=np.nan):
    """Evaluate saved selections using true needs, with one row per model."""
    qualification_col = f"qualified_gagne_{need_threshold}"
    k = int(selection_rate * SAMPLE_SIZE)
    rows = []
    for _, cohort_info in df_people.iterrows():
        seed = cohort_info["seed"]
        iteration = cohort_info["iteration"]

        split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
        split_df = split_df.set_index("person_id")
        cohort = split_df.loc[cohort_info["people"]]

        for algorithm, allocations in algorithm_allocations:
            cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

            # metrics2 expects one selection per model, including repeated sets.
            selected_lists = [
                selected_ids
                for selected_ids, models in zip(cohort_allocations["selected"], cohort_allocations["models"])
                for _ in models
            ]
            if not selected_lists:
                continue
            cohort_stats = cohort_metrics(
                selected_lists,
                cohort.index.to_list(),
                cohort["y"].eq(1).to_numpy(),
            )
            additional_metrics = {name: cohort_stats[name] for name in cohort_metric_columns}

            # Count selected sets, not model rows or allocation_idx values.
            # Sorting makes [person_a, person_b] and [person_b, person_a] identical.
            unique_allocations = {
                tuple(sorted(selected_ids)) for selected_ids in cohort_allocations["selected"]
            }
            model_names = {
                model for models in cohort_allocations["models"] for model in models
            }
            num_unique_allocations = len(unique_allocations)
            num_models = len(model_names)
            unique_allocation_ratio = num_unique_allocations / num_models if num_models else np.nan

            for _, allocation in cohort_allocations.iterrows():
                selected = cohort.loc[allocation["selected"]]

                selected_0 = selected.loc[selected[GROUP_COL] == 0, NEED_COL]
                selected_1 = selected.loc[selected[GROUP_COL] == 1, NEED_COL]

                mean_0_tru = selected_0.mean()
                mean_1_tru = selected_1.mean()

                absolute_disparity_tru = abs(mean_1_tru - mean_0_tru)
                disparity_ratio_tru = np.nan
                if pd.notna(mean_0_tru) and pd.notna(mean_1_tru) and mean_0_tru != 0 and mean_1_tru != 0:
                    disparity_ratio_tru = max(mean_1_tru / mean_0_tru, mean_0_tru / mean_1_tru)

                for model in allocation["models"]:
                    rows.append({
                        "method": METHOD,
                        "algorithm": algorithm,
                        "qualification_column": qualification_col,
                        "seed": seed,
                        "iteration": iteration,
                        "model": model,
                        "alpha": alpha,
                        "beta": beta,
                        "k": k,
                        "k_0": len(selected_0),
                        "k_1": len(selected_1),
                        "n": SAMPLE_SIZE,
                        "tau": need_threshold,
                        "selection_rate": selection_rate,
                        "pred_variant": pred_method,
                        "allocation_epsilon": ALLOCATION_EPS,
                        "rashomon_epsilon": RASHOMON_EPS,
                        "tolerance_type": "add",
                        "expected_utility": selected[model].mean(),
                        "realized_utility": selected["y"].mean(),
                        "absolute_disparity": absolute_disparity_tru,
                        "disparity_ratio": disparity_ratio_tru,
                        "num_models": num_models,
                        "num_unique_allocations": num_unique_allocations,
                        "unique_allocation_ratio": unique_allocation_ratio,
                        **additional_metrics,
                    }) 

    return rows


In [154]:
all_results = []
for PRED_METHOD in PRED_METHODS:
    for NEED_THRESHOLD in NEED_THRESHOLDS:
        df = predictions_for(PRED_METHOD, NEED_THRESHOLD)
        for SELECTION_RATE in tqdm(SELECTION_RATES):
            tag = allocation_tag(PRED_METHOD, NEED_THRESHOLD, SELECTION_RATE)
            df_people = pd.read_csv(ALLOCATION_DIR / f"people_{tag}.csv", converters={"people": literal_eval})
            for algorithm, prefix in [("OPT", "opt"), ("LDA-tru", "lda_tru"), ("LDA-exp", "lda_exp")]:
                allocations = read_allocations(ALLOCATION_DIR / f"{prefix}_{tag}.csv")
                all_results.extend(evaluate_allocations(
                    df, df_people, [(algorithm, allocations)],
                    pred_method=PRED_METHOD, need_threshold=NEED_THRESHOLD,
                    selection_rate=SELECTION_RATE,
                    alpha=0 if algorithm == "LDA-exp" else 1,
                ))

df_results = pd.DataFrame(all_results)

100%|██████████| 1/1 [00:05<00:00,  5.45s/it]


In [155]:
for PRED_METHOD in PRED_METHODS:
    for NEED_THRESHOLD in NEED_THRESHOLDS:
        df = predictions_for(PRED_METHOD, NEED_THRESHOLD)
        for SELECTION_RATE in tqdm(SELECTION_RATES):
            tag = allocation_tag(PRED_METHOD, NEED_THRESHOLD, SELECTION_RATE)
            for REPLACE_ALPHA in REPLACE_ALPHAS:
                suffix = f"{tag}_alpha-{REPLACE_ALPHA}"
                df_people = pd.read_csv(ALLOCATION_DIR / f"people_{suffix}.csv", converters={"people": literal_eval})
                allocations = read_allocations(ALLOCATION_DIR / f"lda_exp_{suffix}.csv")
                all_results.extend(evaluate_allocations(
                    df, df_people, [("LDA-exp", allocations)],
                    pred_method=PRED_METHOD, need_threshold=NEED_THRESHOLD,
                    selection_rate=SELECTION_RATE, alpha=REPLACE_ALPHA,
                ))

df_results = pd.DataFrame(all_results)

100%|██████████| 1/1 [00:09<00:00,  9.37s/it]


### Beta error-shrinkage results

Load the available `lda_exp_*_beta-<value>.csv` and matching `people_*_beta-<value>.csv` files. Rerun this cell and the summary after more beta runs finish; it does not run the allocation algorithms.

`LDA-exp-beta` rows use alpha = 0 and show beta explicitly: beta = 0 uses true needs, and beta = 1 uses predicted needs. Keeping these rows separate preserves both endpoints without pooling them with the baseline runs. Beta is blank (`NaN`) for the existing baseline and alpha rows because it is not the parameter varied in those runs. The existing alpha labels are retained.

All rows use the same metrics, and disparities are evaluated against true needs. The summary reports mean ± standard deviation; consistency and exclusion are averaged once per cohort.

In [156]:
# Replace beta rows when rerunning this cell, so new saved runs can be added safely.
all_results = [row for row in all_results if row["algorithm"] != "LDA-exp-beta"]
for PRED_METHOD in PRED_METHODS:
    for NEED_THRESHOLD in NEED_THRESHOLDS:
        df = predictions_for(PRED_METHOD, NEED_THRESHOLD)
        for SELECTION_RATE in tqdm(SELECTION_RATES):
            tag = allocation_tag(PRED_METHOD, NEED_THRESHOLD, SELECTION_RATE)
            available_runs = beta_runs(tag)
            if not available_runs:
                print(f"No saved beta result pairs found for {tag}.")
            for SHRINK_BETA, people_path, allocation_path in available_runs:
                df_people = pd.read_csv(people_path, converters={"people": literal_eval})
                allocations = read_allocations(allocation_path)
                all_results.extend(evaluate_allocations(
                    df, df_people, [("LDA-exp-beta", allocations)],
                    pred_method=PRED_METHOD, need_threshold=NEED_THRESHOLD,
                    selection_rate=SELECTION_RATE, alpha=0.0, beta=SHRINK_BETA,
                ))

df_results = pd.DataFrame(all_results)
df_results

100%|██████████| 1/1 [00:09<00:00,  9.18s/it]


,method,algorithm,qualification_column,seed,iteration,model,alpha,beta,k,k_0,...,expected_utility,realized_utility,absolute_disparity,disparity_ratio,num_models,num_unique_allocations,unique_allocation_ratio,pairwise_consistency,systemic_exclusion_all,systemic_exclusion_qualified
0,bootstrap,OPT,qualified_gagne_1,0,0,m_1,1.0,NaN,10,9,...,0.980856,1.0,1.777778,1.592593,100,70,0.700000,0.956667,0.80,0.629630
1,bootstrap,OPT,qualified_gagne_1,0,0,m_51,1.0,NaN,10,9,...,0.984786,1.0,1.777778,1.592593,100,70,0.700000,0.956667,0.80,0.629630
2,bootstrap,OPT,qualified_gagne_1,0,0,m_2,1.0,NaN,10,10,...,0.980218,1.0,NaN,NaN,100,70,0.700000,0.956667,0.80,0.629630
3,bootstrap,OPT,qualified_gagne_1,0,0,m_3,1.0,NaN,10,8,...,0.988039,1.0,0.375000,1.083333,100,70,0.700000,0.956667,0.80,0.629630
4,bootstrap,OPT,qualified_gagne_1,0,0,m_9,1.0,NaN,10,8,...,0.978500,1.0,0.375000,1.083333,100,70,0.700000,0.956667,0.80,0.629630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32365,bootstrap,LDA-exp-beta,qualified_gagne_1,4,4,m_96,0.0,1.0,10,4,...,0.988772,1.0,0.583333,1.122807,98,96,0.979592,0.921847,0.76,0.630769
32366,bootstrap,LDA-exp-beta,qualified_gagne_1,4,4,m_97,0.0,1.0,10,5,...,0.983204,1.0,1.400000,1.350000,98,96,0.979592,0.921847,0.76,0.630769
32367,bootstrap,LDA-exp-beta,qualified_gagne_1,4,4,m_98,0.0,1.0,10,4,...,0.988464,1.0,1.750000,1.368421,98,96,0.979592,0.921847,0.76,0.630769
32368,bootstrap,LDA-exp-beta,qualified_gagne_1,4,4,m_99,0.0,1.0,10,4,...,0.991730,1.0,2.083333,1.438596,98,96,0.979592,0.921847,0.76,0.630769


In [157]:
columns = ["realized_utility", "expected_utility", "absolute_disparity", "disparity_ratio", "k_0", "k_1"]
summary_keys = ["selection_rate", "k", "n", "pred_variant", "algorithm", "alpha", "beta"]
grouped = df_results.groupby(summary_keys, dropna=False)[columns].agg(["mean", "std"])

df_cohort_metrics = df_results.drop_duplicates(summary_keys + ["seed", "iteration"])
cohort_grouped = df_cohort_metrics.groupby(summary_keys, dropna=False)[cohort_metric_columns].agg(["mean", "std"])
grouped = pd.concat([grouped, cohort_grouped], axis=1)
columns = columns + cohort_metric_columns
summary = pd.DataFrame(index=grouped.index)

for col in columns:
    summary[col] = grouped.apply(lambda row: f"{row[(col, 'mean')]:.2f} \u00b1 {row[(col, 'std')]:.2f}", axis=1)
summary

realized_utility  \
selection_rate k  n   pred_variant algorithm    alpha beta                    
0.1            10 100 average      LDA-exp      0.00  NaN       0.96 ± 0.06   
                                                0.25  NaN       0.97 ± 0.06   
                                                0.50  NaN       0.97 ± 0.07   
                                                0.75  NaN       0.96 ± 0.06   
                                                1.00  NaN       0.96 ± 0.06   
                                   LDA-exp-beta 0.00  0.00      0.96 ± 0.06   
                                                      0.25      0.96 ± 0.07   
                                                      0.50      0.96 ± 0.07   
                                                      0.75      0.96 ± 0.07   
                                                      1.00      0.96 ± 0.06   
                                   LDA-tru      1.00  NaN       0.96 ± 0.06   
                                   OPT          1.00  NaN       0.98 ± 0.04   

                                                           expected_utility  \
selection_rate k  n   pred_variant algorithm    alpha beta                    
0.1            10 100 average      LDA-exp      0.00  NaN       0.97 ± 0.03   
                                                0.25  NaN       0.97 ± 0.03   
                                                0.50  NaN       0.97 ± 0.03   
                                                0.75  NaN       0.97 ± 0.03   
                                                1.00  NaN       0.97 ± 0.03   
                                   LDA-exp-beta 0.00  0.00      0.97 ± 0.03   
                                                      0.25      0.97 ± 0.03   
                                                      0.50      0.97 ± 0.03   
                                                      0.75      0.97 ± 0.03   
                                                      1.00      0.97 ± 0.03   
                                   LDA-tru      1.00  NaN       0.97 ± 0.03   
                                   OPT          1.00  NaN       0.99 ± 0.00   

                                                           absolute_disparity  \
selection_rate k  n   pred_variant algorithm    alpha beta                      
0.1            10 100 average      LDA-exp      0.00  NaN         1.65 ± 1.15   
                                                0.25  NaN         1.41 ± 1.07   
                                                0.50  NaN         1.11 ± 0.95   
                                                0.75  NaN         0.72 ± 0.79   
                                                1.00  NaN         0.24 ± 0.72   
                                   LDA-exp-beta 0.00  0.00        0.24 ± 0.72   
                                                      0.25        0.50 ± 0.69   
                                                      0.50        0.79 ± 0.77   
                                                      0.75        1.17 ± 0.90   
                                                      1.00        1.65 ± 1.15   
                                   LDA-tru      1.00  NaN         0.24 ± 0.72   
                                   OPT          1.00  NaN         1.97 ± 1.41   

                                                           disparity_ratio  \
selection_rate k  n   pred_variant algorithm    alpha beta                   
0.1            10 100 average      LDA-exp      0.00  NaN      1.62 ± 0.74   
                                                0.25  NaN      1.54 ± 0.72   
                                                0.50  NaN      1.46 ± 0.70   
                                                0.75  NaN      1.26 ± 0.47   
                                                1.00  NaN      1.11 ± 0.38   
                                   LDA-exp-beta 0.00  0.00     1.11 ± 0.38   
                                                      0.25     1.17 ± 0.35   
        

### Absolute disparity: alpha and beta experiments

The panels share a y-axis and use the same model-level mean and sample standard deviation as the results table. Alpha replaces a fraction of predicted needs with true needs; beta scales the prediction error. **More correction means higher alpha or lower beta.** Only saved experiment points are plotted; alpha = 0 comes from the LDA-exp baseline.

Dashed reference lines show the mean for LDA-exp, LDA-tru, and OPT. Lighter bands show **mean ± 1 standard deviation**, including for the references, with every lower bound clipped at zero. These bands show variation across evaluated allocations, not confidence intervals. All disparities are evaluated using true needs. Click legend entries to hide or show a series and its band.

In [162]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Keep experiment configurations separate, and retain baseline rows with NaN beta.
plot_config_columns = ["selection_rate", "k", "n", "pred_variant", "tau"]
disparity_plot_stats = (
    df_results.groupby(plot_config_columns + ["algorithm", "alpha", "beta"], dropna=False)["absolute_disparity"]
    .agg(["mean", "std"])
    .reset_index()
)
disparity_plot_stats["lower"] = (disparity_plot_stats["mean"] - disparity_plot_stats["std"]).clip(lower=0)
disparity_plot_stats["upper"] = disparity_plot_stats["mean"] + disparity_plot_stats["std"]


def add_disparity_series(fig, *, x, mean, std, lower, upper, name, color,
                         column, dash="solid", marker=None, opacity=0.18, showlegend=True):
    # Every band uses clipped bounds, including the horizontal reference bands.
    red, green, blue = (int(color[index:index + 2], 16) for index in (1, 3, 5))
    for bounds, fill in [(lower, None), (upper, "tonexty")]:
        fig.add_trace(go.Scatter(
            x=list(x), y=list(bounds), mode="lines", line=dict(width=0),
            fill=fill, fillcolor=f"rgba({red},{green},{blue},{opacity})",
            name=name, legendgroup=name, showlegend=False, hoverinfo="skip",
        ), row=1, col=column)
    fig.add_trace(go.Scatter(
        x=list(x), y=list(mean), customdata=list(std),
        mode="lines+markers" if marker else "lines",
        line=dict(color=color, width=2.6 if marker else 1.8, dash=dash),  # noqa: C408
        marker=dict(symbol=marker or "circle", size=8),  # noqa: C408
        name=name, legendgroup=name, showlegend=showlegend,
        hovertemplate=(
            ("Parameter: %{x:g}<br>" if marker else "")
            + "Mean disparity: %{y:.3f}<br>SD: %{customdata:.3f}<extra>%{fullData.name}</extra>"
        ),
    ), row=1, col=column)


reference_styles = [
    ("LDA-exp", "#7A5195", "dash"),
    ("LDA-tru", "#009E73", "dot"),
    ("OPT", "#242424", "dashdot"),
]
disparity_figures = []
for config, stats in disparity_plot_stats.groupby(plot_config_columns, dropna=False, sort=True):
    rate, k, n, pred_variant, tau = config
    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08,
        subplot_titles=("Alpha: replace predicted needs", "Beta: shrink prediction errors"),
    )

    # Alpha = 0 identifies the original LDA-exp baseline, not the replacement runs.
    references = stats.loc[
        stats["beta"].isna()
        & (stats["algorithm"].isin(["OPT", "LDA-tru"])
           | (stats["algorithm"].eq("LDA-exp") & stats["alpha"].eq(0)))
    ].set_index("algorithm")
    for algorithm, color, dash in reference_styles:
        if algorithm not in references.index:
            continue
        reference = references.loc[algorithm]
        for column in [1, 2]:
            add_disparity_series(
                fig, x=[0, 1], mean=[reference["mean"]] * 2, std=[reference["std"]] * 2,
                lower=[reference["lower"]] * 2, upper=[reference["upper"]] * 2,
                name=algorithm, color=color, dash=dash, column=column,
                opacity=0.06, showlegend=column == 1,
            )

    experiments = [
        (1, "alpha", "Alpha replacement", "#0072B2", "circle",
         stats["algorithm"].eq("LDA-exp") & stats["beta"].isna()),
        (2, "beta", "Beta shrinkage", "#D55E00", "diamond",
         stats["algorithm"].eq("LDA-exp-beta") & stats["alpha"].eq(0)),
    ]
    for column, parameter, name, color, marker, mask in experiments:
        curve = stats.loc[mask].sort_values(parameter)
        if curve.empty:
            fig.add_annotation(text="No saved results", x=0.5, y=0.5,
                               xref=f"x{column if column > 1 else ''} domain",
                               yref=f"y{column if column > 1 else ''} domain", showarrow=False)
            continue
        add_disparity_series(
            fig, x=curve[parameter], mean=curve["mean"], std=curve["std"],
            lower=curve["lower"], upper=curve["upper"], name=name,
            color=color, marker=marker, column=column,
        )

    y_max = max(float(stats["upper"].fillna(stats["mean"]).max()), 0.1) * 1.05
    fig.update_xaxes(range=[-0.02, 1.02], tickvals=np.linspace(0, 1, 6).tolist(), showgrid=False)
    fig.update_xaxes(title_text="α — Fraction replaced with true needs", row=1, col=1)
    fig.update_xaxes(title_text="β — Fraction of prediction error retained", row=1, col=2)
    fig.update_yaxes(range=[0, y_max], gridcolor="#E5E7EB", zeroline=False)
    fig.update_yaxes(title_text="Absolute disparity", row=1, col=1)
    fig.update_layout(
        template="plotly_white", height=600,
        font=dict(size=13, family="iosevka"), margin=dict(l=75, r=30, t=115, b=100),  # noqa: C408
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.22, yanchor="top", groupclick="togglegroup"),  # noqa: C408
        hovermode="closest",
    )
    disparity_figures.append(fig)
    fig.show()
